# Qwen2.5-3B complete CrashDiag pipeline

Run the cells top-to-bottom. This generates a bounded dataset, evaluates the base model, runs one-epoch SFT, evaluates SFT, runs GRPO, evaluates GRPO, displays every graph, and uploads each stage to the HF bucket.

In [ ]:
import os, subprocess, sys
from pathlib import Path

LAUNCH_DIR = Path.cwd().resolve()
subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv>=1,<2'], check=True)
from dotenv import load_dotenv
ENV_FILE = Path(os.environ.get('CRASHDIAG_ENV_FILE', LAUNCH_DIR / 'env.txt')).expanduser()
if not ENV_FILE.is_absolute(): ENV_FILE = (LAUNCH_DIR / ENV_FILE).resolve()
if ENV_FILE.is_file(): load_dotenv(ENV_FILE, override=True)

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None
KAGGLE_SECRET_ALIASES = {
    'HF_TOKEN': 'HF_TOKEN',
    'CRASHDIAG_DATASET_RUN_ID': 'CRASHDIAG_DATASET_RUN_ID', 'DATASET_RUN_ID': 'CRASHDIAG_DATASET_RUN_ID',
    'CRASHDIAG_SANDBOX_URL': 'CRASHDIAG_SANDBOX_URL',
    'CRASHDIAG_API_TOKEN': 'CRASHDIAG_API_TOKEN', 'CRASHDIAG_SANDBOX_TOKEN': 'CRASHDIAG_SANDBOX_TOKEN',
    'CRASHDIAG_SOURCE_COMMIT': 'CRASHDIAG_SOURCE_COMMIT', 'SOURCE_COMMIT': 'CRASHDIAG_SOURCE_COMMIT',
}
loaded_kaggle_secrets, kaggle_secret_errors = [], {}
if UserSecretsClient is not None:
    client = UserSecretsClient()
    for secret_name, env_name in KAGGLE_SECRET_ALIASES.items():
        if os.environ.get(env_name): continue
        try: value = client.get_secret(secret_name)
        except Exception as exc:
            kaggle_secret_errors[secret_name] = f'{type(exc).__name__}: {exc}'
            continue
        if value:
            os.environ[env_name] = value
            loaded_kaggle_secrets.append(secret_name)
print('loaded Kaggle secret names:', loaded_kaggle_secrets or 'none')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

REPO_URL = os.environ.get('CRASHDIAG_REPO_URL', 'https://github.com/Indium-AI-Labs/CrashDiag.git')
SOURCE_COMMIT = os.environ.get('CRASHDIAG_SOURCE_COMMIT', 'main')
WORKDIR = Path(os.environ.get('CRASHDIAG_WORKDIR', LAUNCH_DIR / 'CrashDiag-runtime')).expanduser().resolve()
if (WORKDIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(WORKDIR), 'fetch', 'origin', 'main'], check=True)
elif WORKDIR.exists() and any(WORKDIR.iterdir()):
    raise RuntimeError(f'CRASHDIAG_WORKDIR is not an empty Git checkout: {WORKDIR}')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)
subprocess.run(['git', '-C', str(WORKDIR), 'checkout', SOURCE_COMMIT], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'bitsandbytes'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train]'], check=True)
print('env_file=', ENV_FILE if ENV_FILE.is_file() else 'not present (runtime/Kaggle secrets)')
print('checked_out_source_commit=', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MODEL_SLUG = "qwen2.5_3b"
BUCKET_ID = "devaanshpa/CrashDiag"
TRAIN_SAMPLES_PER_FAULT = int(os.environ.get('CRASHDIAG_TRAIN_SAMPLES_PER_FAULT', '128'))
EVAL_SAMPLES_PER_FAULT = int(os.environ.get('CRASHDIAG_EVAL_SAMPLES_PER_FAULT', '16'))
TRAIN_FILE, EVAL_FILE = 'grpo_train.jsonl', 'grpo_eval.jsonl'
def ist_run_id(stage):
    return datetime.now(ZoneInfo('Asia/Kolkata')).strftime('%Y%m%dT%H%M%SIST') + f'-{MODEL_SLUG}-{stage}'
DATASET_RUN_ID = os.environ.get('CRASHDIAG_DATASET_RUN_ID', '').strip() or ist_run_id('dataset')
BASE_EVAL_RUN_ID = os.environ.get('CRASHDIAG_BASE_EVAL_RUN_ID', '').strip() or ist_run_id('base-eval')
SFT_RUN_ID = os.environ.get('CRASHDIAG_SFT_RUN_ID', '').strip() or ist_run_id('sft')
SFT_EVAL_RUN_ID = os.environ.get('CRASHDIAG_SFT_EVAL_RUN_ID', '').strip() or ist_run_id('sft-eval')
GRPO_RUN_ID = os.environ.get('CRASHDIAG_GRPO_RUN_ID', '').strip() or ist_run_id('grpo')
GRPO_EVAL_RUN_ID = os.environ.get('CRASHDIAG_GRPO_EVAL_RUN_ID', '').strip() or ist_run_id('grpo-eval')
SANDBOX_TOKEN = os.environ.get('CRASHDIAG_API_TOKEN') or os.environ.get('CRASHDIAG_SANDBOX_TOKEN', '')
print(f'base_model={BASE_MODEL}')
print(f'train_samples_per_fault={TRAIN_SAMPLES_PER_FAULT}, eval_samples_per_fault={EVAL_SAMPLES_PER_FAULT}')
print(f'dataset_run_id={DATASET_RUN_ID}')


In [ ]:
from training.generate_dataset import generate_datasets
from training.artifacts import ArtifactConfig, ArtifactUploader, runtime_metadata
print('Generating and mechanically validating dataset...')
counts = generate_datasets(train_samples_per_fault=TRAIN_SAMPLES_PER_FAULT, eval_samples_per_fault=EVAL_SAMPLES_PER_FAULT, seed=42)
print(f"dataset rows: sft_train={counts['sft_train']}, sft_eval={counts['sft_eval']}, grpo_train={counts['grpo_train']}, grpo_eval={counts['grpo_eval']}")
print(f"expected SFT optimizer steps: {(counts['sft_train'] + 63) // 64} (batch=8, accumulation=8, one epoch)")
token = os.environ['HF_TOKEN']
uploader = ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID, token=token, policy='required'))
source_commit = str(runtime_metadata().get('git_commit', 'unknown'))
uploader.start_run({'entrypoint': 'notebooks.run_all', 'source_commit': source_commit})
uploader.start_stage('datasets', {'source_commit': source_commit, 'seed': 42, 'train_samples_per_fault': TRAIN_SAMPLES_PER_FAULT, 'eval_samples_per_fault': EVAL_SAMPLES_PER_FAULT, 'schema_version': 5, 'curriculum_version': 5})
uploader.upload_files(['data/sft_train.jsonl', 'data/sft_eval.jsonl', 'data/grpo_train.jsonl', 'data/grpo_eval.jsonl', 'data/grpo_summary.json'], 'datasets', metadata={'source_commit': source_commit, 'mechanically_validated': True, 'grpo_targets_included': False, 'curricula': ['v5']})
uploader.complete_run({'stages': ['datasets']})
print('datasets uploaded:', uploader.remote_uri('datasets'))


In [ ]:
from pathlib import Path
from training.artifacts import ArtifactConfig, ArtifactUploader
DATASET_DIR = Path('artifacts/datasets')
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID, token=os.environ['HF_TOKEN'])).download_stage('datasets', DATASET_DIR)
assert (DATASET_DIR / EVAL_FILE).is_file() and (DATASET_DIR / 'sft_train.jsonl').is_file()
print('datasets ready:', DATASET_DIR)


In [ ]:
from training.evaluate_jsonl import main as evaluate_main
print('=== BASE EVALUATION ===', flush=True)
exit_code = evaluate_main(['--model', BASE_MODEL, '--dataset', str(DATASET_DIR / EVAL_FILE), '--output-dir', 'outputs/base-eval', '--load-in-4bit', '--precision', 'bf16', '--max-new-tokens', '64', '--sandbox-url', os.environ['CRASHDIAG_SANDBOX_URL'], '--sandbox-token', SANDBOX_TOKEN, '--artifact-bucket', BUCKET_ID, '--run-id', BASE_EVAL_RUN_ID, '--artifact-stage', 'base-eval', '--no-few-shot'])
if exit_code: raise RuntimeError(f'base eval failed: {exit_code}')
print('base eval complete:', BASE_EVAL_RUN_ID)


In [ ]:
import subprocess, sys
print('=== ONE-EPOCH SFT ===', flush=True)
command = [sys.executable, '-m', 'accelerate.commands.launch', '--num_processes', '1', '--num_machines', '1', '--mixed_precision', 'bf16', '--dynamo_backend', 'no', '-m', 'training.sft', '--model', BASE_MODEL, '--dataset', str(DATASET_DIR / 'sft_train.jsonl'), '--eval-dataset', str(DATASET_DIR / 'sft_eval.jsonl'), '--output-dir', 'outputs/sft', '--epochs', '1', '--batch-size', '8', '--eval-batch-size', '8', '--gradient-accumulation-steps', '8', '--max-length', '512', '--learning-rate', '2e-4', '--lora-rank', '16', '--lora-alpha', '32', '--load-in-4bit', '--precision', 'bf16', '--report-to', 'none', '--artifact-bucket', BUCKET_ID, '--run-id', SFT_RUN_ID]
subprocess.run(command, check=True)
print('SFT complete:', SFT_RUN_ID)


In [ ]:
print('=== SFT EVALUATION ===', flush=True)
exit_code = evaluate_main(['--model', 'outputs/sft', '--dataset', str(DATASET_DIR / EVAL_FILE), '--output-dir', 'outputs/sft-eval', '--load-in-4bit', '--precision', 'bf16', '--max-new-tokens', '64', '--sandbox-url', os.environ['CRASHDIAG_SANDBOX_URL'], '--sandbox-token', SANDBOX_TOKEN, '--artifact-bucket', BUCKET_ID, '--run-id', SFT_EVAL_RUN_ID, '--artifact-stage', 'sft-eval', '--no-few-shot'])
if exit_code: raise RuntimeError(f'SFT eval failed: {exit_code}')
print('SFT eval complete:', SFT_EVAL_RUN_ID)


In [ ]:
print('=== GRPO ===', flush=True)
command = [sys.executable, '-m', 'accelerate.commands.launch', '--num_processes', '1', '--num_machines', '1', '--mixed_precision', 'bf16', '--dynamo_backend', 'no', '-m', 'training.grpo', '--model', 'outputs/sft', '--train-file', str(DATASET_DIR / TRAIN_FILE), '--eval-file', str(DATASET_DIR / EVAL_FILE), '--output-dir', 'outputs/grpo', '--load-in-4bit', '--precision', 'bf16', '--batch-size', '2', '--gradient-accumulation-steps', '4', '--num-generations', '2', '--max-prompt-length', '1024', '--max-completion-length', '64', '--max-steps', os.environ.get('CRASHDIAG_GRPO_MAX_STEPS', '96'), '--artifact-bucket', BUCKET_ID, '--run-id', GRPO_RUN_ID, '--artifact-stage', 'grpo', '--sandbox-url', os.environ['CRASHDIAG_SANDBOX_URL'], '--sandbox-token', SANDBOX_TOKEN]
subprocess.run(command, check=True)
print('GRPO complete:', GRPO_RUN_ID)


In [ ]:
print('=== GRPO EVALUATION ===', flush=True)
exit_code = evaluate_main(['--model', 'outputs/grpo', '--dataset', str(DATASET_DIR / EVAL_FILE), '--output-dir', 'outputs/grpo-eval', '--load-in-4bit', '--precision', 'bf16', '--max-new-tokens', '64', '--sandbox-url', os.environ['CRASHDIAG_SANDBOX_URL'], '--sandbox-token', SANDBOX_TOKEN, '--artifact-bucket', BUCKET_ID, '--run-id', GRPO_EVAL_RUN_ID, '--artifact-stage', 'grpo-eval', '--no-few-shot'])
if exit_code: raise RuntimeError(f'GRPO eval failed: {exit_code}')
print('GRPO eval complete:', GRPO_EVAL_RUN_ID)


In [ ]:
from IPython.display import SVG, display
print('=== REPORTS (also uploaded by each stage) ===')
for stage in ('base-eval', 'sft', 'sft-eval', 'grpo', 'grpo-eval'):
    reports = Path("outputs") / stage / "reports"
    charts = sorted(reports.glob('*.svg')) if reports.is_dir() else []
    print(f'== {stage}: {len(charts)} charts ==')
    for chart in charts: display(SVG(filename=str(chart)))
print('\n=== RUN ALL COMPLETE ===')
for name, value in [('dataset', DATASET_RUN_ID), ('base eval', BASE_EVAL_RUN_ID), ('sft', SFT_RUN_ID), ('sft eval', SFT_EVAL_RUN_ID), ('grpo', GRPO_RUN_ID), ('grpo eval', GRPO_EVAL_RUN_ID)]: print(f'{name}: {value}')
